# 01 Data audit: first look at the raw data

The exploration that came first. The reproducible version, with every number used later, is
[reports/data_audit.md](../reports/data_audit.md), written by `python -m scorecard.data.audit`.
Raw rows are deliberately not printed: Kaggle's licence forbids redistributing the data
(see [data/README.md](../data/README.md)).

## Hypotheses to check against the data
- **Bad rate:** expected to be low (single digits), because these are borrowers a lender already approved, and 90+ days past due is a severe outcome.
- **Missing values:** expected in self-reported fields such as `MonthlyIncome` and `NumberOfDependents`.
- **Extremes:** utilisation (balance / limit) should mostly sit in 0-1; values far above 1 and implausible ages would point to data errors.

In [ ]:
import sys

import numpy as np
import pandas as pd

from scorecard.data.load import PAST_DUE_COLUMNS, TARGET, load_raw

print(sys.executable, np.__version__, pd.__version__)

## Load

`load_raw()` is the loader the pipeline uses, so this notebook sees exactly the same data. It drops
`Unnamed: 0`, the CSV's row number. That matters for the duplicates check below: with a unique
row number in every row, no two rows could ever be identical.

In [ ]:
df = load_raw()
df.shape

In [ ]:
df.dtypes

## Bad rate

In [ ]:
df[TARGET].value_counts(normalize=True)

## Missing values

In [ ]:
missing = pd.DataFrame({"missing": df.isna().sum(), "share": df.isna().mean()})
missing[missing["missing"] > 0]

## Duplicates

In [ ]:
# Count the 2nd, 3rd... copies of identical rows (all columns, target included).
df.duplicated().sum()

## Extremes

In [ ]:
df.describe().T

In [ ]:
pd.Series(
    {
        "utilisation > 1 (share)": (df["RevolvingUtilizationOfUnsecuredLines"] > 1).mean(),
        "age < 18 (rows)": (df["age"] < 18).sum(),
        "age > 100 (rows)": (df["age"] > 100).sum(),
        "past-due count >= 96 (rows)": (df[PAST_DUE_COLUMNS] >= 96).any(axis=1).sum(),
    }
)

## What the hypotheses showed

- **Bad rate:** 6.68%, single digits as expected. The imbalance is handled by stratifying the split and judging with AUC / Gini / KS (D-011, D-012).
- **Missing values:** only `MonthlyIncome` (19.8%) and `NumberOfDependents` (2.6%), both self-reported, as expected. They stay missing and get their own bin (D-003, D-005).
- **Extremes:** confirmed. Utilisation goes far above 1, one row has age 0, and the past-due counts hold 96 / 98 codes that are placeholders, not counts (D-006, D-007, D-008).
- **Not expected:** 609 exact duplicate rows with no customer ID to resolve them (D-009). Their effect on AUC was measured at evaluation: [reports/duplicate_check.md](../reports/duplicate_check.md).